In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the two sliders. The red stems are the DFT bins and the gray curve
is the smooth spectrum they sample, with the two true tones dotted. A
longer duration packs the bins closer together and separates the two
peaks. The sample rate changes how many samples there are, but not how
far apart the bins sit.

In [ ]:
# hide
# autorun
F1, F2 = 440.0, 460.0               # two tones 20 Hz apart
T0, FS0 = 0.2, 8000.0               # starting parameters
F_LO, F_HI = 360.0, 540.0
NPAD = 1 << 15

def analyze(T, fs, F1=F1, F2=F2, F_LO=F_LO, F_HI=F_HI, NPAD=NPAD):
    N = int(round(T * fs))
    n = np.arange(N)
    x = np.sin(2 * np.pi * F1 * n / fs) + np.sin(2 * np.pi * F2 * n / fs)
    # the bins, and the same spectrum zero-padded into a smooth curve that
    # passes through every bin
    X = np.abs(np.fft.rfft(x))
    fk = np.arange(len(X)) * fs / N
    Xp = np.abs(np.fft.rfft(x, NPAD))
    fp = np.arange(len(Xp)) * fs / NPAD
    keep, keep_p = (fk >= F_LO) & (fk <= F_HI), (fp >= F_LO) & (fp <= F_HI)
    peak = Xp[keep_p].max()
    return fk[keep], X[keep] / peak, fp[keep_p], Xp[keep_p] / peak, N

def stems(fk, mag):
    xs = np.repeat(fk, 3).astype(object)
    ys = np.column_stack([np.zeros_like(mag), mag, mag]).ravel().astype(object)
    xs[2::3], ys[2::3] = None, None
    return xs, ys

def figure():
    fig = go.Figure()
    fk, mag, fp, magp, _ = analyze(T0, FS0)
    sx, sy = stems(fk, mag)
    fig.add_scatter(x=fp, y=magp, mode="lines", line=dict(color=STEEL, width=1.8))
    fig.add_scatter(x=[F1, F1, None, F2, F2, None], y=[0, 1.08, None, 0, 1.08, None],
                    mode="lines", line=dict(color=IRON, width=1.2, dash="dot"))
    fig.add_scatter(x=sx, y=sy, mode="lines", line=dict(color=RED, width=1.6))
    fig.add_scatter(x=fk, y=mag, mode="markers", marker=dict(color=RED, size=6))
    fig.update_xaxes(range=[F_LO, F_HI], title_text="Frequency (Hz)", fixedrange=True)
    fig.update_yaxes(range=[0, 1.12], title_text="Magnitude", fixedrange=True)
    return fig

def controls(fig):
    dur = widgets.FloatSlider(description="Duration T (s)", min=0.02, max=0.4,
                              value=T0, step=0.005, readout_format=".3f")
    fs = widgets.FloatSlider(description="Sample rate f_s (Hz)", min=4000,
                             max=16000, value=FS0, step=1000, readout_format=".0f")
    readout = widgets.HTML()

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def update(T, fs, analyze=analyze, stems=stems, readout=readout):
        fk, mag, fp, magp, N = analyze(T, fs)
        sx, sy = stems(fk, mag)
        with fig.batch_update():
            fig.data[0].x, fig.data[0].y = fp, magp
            fig.data[2].x, fig.data[2].y = sx, sy
            fig.data[3].x, fig.data[3].y = fk, mag
        readout.value = (f"<span style='font-size:0.9em'>N = {N:,} samples "
                         f"&nbsp;·&nbsp; bin spacing f<sub>s</sub> / N = "
                         f"{fs / N:.1f} Hz</span>")

    widgets.interactive_output(update, {"T": dur, "fs": fs})
    return widgets.VBox([dur, fs, readout])

icm_plotly.show(figure, controls)